In [2]:
import os
import argparse
import json
from tqdm import tqdm
import random
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr
import numpy as np

In [3]:
parser = argparse.ArgumentParser()
args = parser.parse_args([])
args.seed = 42
# ------------ Mistral---------------
args.number_type = "decimal"
args.data_path = f"../embeddings_arxiv/Mistral-7B-v0.1/{args.number_type}"
args.num_layers = 32
offset_type = "offset_0"
print(args)

Namespace(seed=42, number_type='decimal', data_path='../embeddings_arxiv/Mistral-7B-v0.1/decimal', num_layers=32)


In [4]:
metadata_path = os.path.join(args.data_path, 'metadata.jsonl')
metadata = []
with open(metadata_path, 'r') as f:
    for line in f:
        metadata.append(json.loads(line))

print(f"Loaded {len(metadata)} samples")

Loaded 5000 samples


In [5]:
# Extract numeric values from metadata
org_values = []
for i, meta in enumerate(metadata):
    val = meta.get('numeric_value') or meta.get('value')
    if val is not None:
        org_values.append(float(val))
    else:
        print(f"Skipping invalid numeric value in sample {i}")
org_values = np.array(org_values)

In [6]:
org_embeddings = []
for layer_idx in tqdm(range(args.num_layers), desc=f"Loading embeddings"):
    embed_path = os.path.join(args.data_path, offset_type, f'layer_{layer_idx+1}.embeds')
    embeddings = []
    with open(embed_path, 'rb') as f:
        while True:
            try:
                embeddings.append(np.load(f))
            except EOFError:
                break
    embeddings = np.array(embeddings)
    org_embeddings.append(embeddings)

Loading embeddings: 100%|██████████| 32/32 [00:11<00:00,  2.80it/s]


In [7]:
for embeddings in org_embeddings:
    assert(len(embeddings) == len(org_values))

np.random.seed(args.seed)
random.seed(args.seed)

# Filter non-zero values
non_zero_idx = np.where(org_values != 0)[0]
values = org_values[non_zero_idx]
embeddings = [emb[non_zero_idx] for emb in org_embeddings]

# shuffle the data
shuffle_idx = np.random.permutation(len(values))
embeddings = [emb[shuffle_idx] for emb in embeddings]
values = values[shuffle_idx]
log_values = np.log2(values)

# Train-val-test split (80-10-10)
val_split_idx = int(0.8 * len(values))
test_split_idx = int(0.9 * len(values))
X_train_all = [emb[:val_split_idx] for emb in embeddings] # all denotes all the layers
X_val_all = [emb[val_split_idx:test_split_idx] for emb in embeddings]
X_test_all = [emb[test_split_idx:] for emb in embeddings]
y_train = log_values[:val_split_idx]
y_val = log_values[val_split_idx:test_split_idx]
y_test = log_values[test_split_idx:]
print(f"Using {len(values)} samples | Train: {X_train_all[0].shape} | Val: {X_val_all[0].shape} | Test: {X_test_all[0].shape}")

Using 5000 samples | Train: (4000, 4096) | Val: (500, 4096) | Test: (500, 4096)


In [8]:
def evaluate_metrics(y_true, y_pred, log_space=True):
    """Calculate evaluation metrics including relative error."""
    r2 = r2_score(y_true, y_pred)
    pearson = pearsonr(y_true, y_pred)[0]
    mse = mean_squared_error(y_true, y_pred)
    
    if log_space:
        # Calculate relative error: |1 - exp2(log_pred - log_true)|
        relative_error = np.abs(1 - np.exp2(y_pred - y_true))
        mean_relative_error = np.mean(relative_error)
        quantiles = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99]
        quantiles_relative_error = np.quantile(relative_error, quantiles)
        quantiles_relative_error_str = [f"{q*100}%: {qe}" for q, qe in zip(quantiles, quantiles_relative_error)]

        # Approximate accuracy (1% tolerance) in original space
        y_true_orig = np.exp2(y_true)
        y_pred_orig = np.exp2(y_pred)
        tolerance = 0.01 * y_true_orig
        aacc = np.mean(np.abs(y_pred_orig - y_true_orig) <= tolerance)
    else:
        # Calculate relative error directly
        relative_error = np.abs((y_pred - y_true) / y_true)
        mean_relative_error = np.mean(relative_error)
        
        # Approximate accuracy (1% tolerance)
        tolerance = 0.01 * y_true
        aacc = np.mean(np.abs(y_pred - y_true) <= tolerance)
    
    return {
        'r2': r2,
        'pearson': pearson, 
        'mse': mse,
        'aacc': aacc,
        'mean_relative_error': mean_relative_error,
        'quantiles_relative_error': quantiles_relative_error_str
    }

In [9]:
offset_models = []
offset_results = []

for layer_idx in tqdm(range(args.num_layers), desc=f"Training {offset_type}"):
    X_train = X_train_all[layer_idx]
    X_val = X_val_all[layer_idx]
    
    model = Ridge(alpha=0.1)
    model.fit(X_train, y_train)

    # Evaluate
    y_val_pred = model.predict(X_val)
    metrics = evaluate_metrics(y_val, y_val_pred, log_space=True)
    metrics['layer'] = layer_idx + 1
    offset_models.append(model)
    offset_results.append(metrics)


Training offset_0: 100%|██████████| 32/32 [00:23<00:00,  1.37it/s]


In [10]:
RE_results = []
for layer_idx in range(args.num_layers):
    y_val_pred = offset_models[layer_idx].predict(X_val_all[layer_idx])
    y_test_pred = offset_models[layer_idx].predict(X_test_all[layer_idx])
    RE_results.append({
        'val_median_RE_real': np.median(np.abs(np.exp2(y_val) - np.exp2(y_val_pred)) / np.abs(np.exp2(y_val))),
        'val_RE_median_symmetric': np.exp2(np.median(np.abs(y_val_pred-y_val))) - 1,
        'val_median_RE_symmetric': np.median(np.exp2((np.abs(y_val_pred-y_val))) - 1),
        'test_median_RE_real': np.median(np.abs(np.exp2(y_test) - np.exp2(y_test_pred)) / np.abs(np.exp2(y_test))),
        'test_RE_median_symmetric': np.exp2(np.median(np.abs(y_test_pred-y_test))) - 1,
        'test_median_RE_symmetric': np.median(np.exp2((np.abs(y_test_pred-y_test))) - 1)
    })

In [11]:
VAL_METRICS = [
    "val_median_RE_real",
    "val_RE_median_symmetric",
    "val_median_RE_symmetric",
]

TEST_METRICS = [
    "test_median_RE_real",
    "test_RE_median_symmetric",
    "test_median_RE_symmetric",
]

# map val metric -> corresponding test metric
VAL_TO_TEST = {
    "val_median_RE_real": "test_median_RE_real",
    "val_RE_median_symmetric": "test_RE_median_symmetric",
    "val_median_RE_symmetric": "test_median_RE_symmetric",
}

print("Best layer selected by VAL metric; report VAL + TEST at that layer (lower is better)\n")

for val_metric in VAL_METRICS:
    test_metric = VAL_TO_TEST[val_metric]

    val_vals = np.array([d[val_metric] for d in RE_results], dtype=float)
    best_layer = int(np.argmin(val_vals))

    best_val = RE_results[best_layer][val_metric]
    best_test = RE_results[best_layer][test_metric]

    print(
        f"select by {val_metric}: "
        f"layer={best_layer}, val={best_val:.6g}, test={best_test:.6g}"
    )


Best layer selected by VAL metric; report VAL + TEST at that layer (lower is better)

select by val_median_RE_real: layer=3, val=0.173861, test=0.17357
select by val_RE_median_symmetric: layer=3, val=0.188778, test=0.189802
select by val_median_RE_symmetric: layer=3, val=0.188778, test=0.189803
